# Stage 0 => Dataset Creation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_PATH = "/content/drive/MyDrive/VR/Dataset"

# Extract Images

In [ ]:
import zipfile
import os

IMG_ZIP = "/content/drive/MyDrive/VR/Dataset/img.zip"
EXTRACT_PATH = "/content/img"

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(IMG_ZIP, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

This tells you:
- which images go to train
- which go to query
- which go to gallery

In [ ]:
eval_file = BASE_PATH + "/list_eval_partition.txt"

In [ ]:
with open(eval_file, 'r') as f:
    lines = f.readlines()

print(lines[:10])  # check first 10 lines

# Create train / query / gallery

In [ ]:
data = []

for line in lines[2:]:  # skip header
    parts = line.strip().split()

    if len(parts) < 3:
        continue

    img_path = parts[0]
    item_id = parts[1]
    split = parts[2]

    data.append((img_path, item_id, split))

print(data[:5])

In [ ]:
import os

DATASET_PATH = "/content/dataset"

for split in ["train", "query", "gallery"]:
    os.makedirs(os.path.join(DATASET_PATH, split), exist_ok=True)

In [ ]:
import shutil
from tqdm import tqdm

mapping = []

for img_path, item_id, split in tqdm(data):

    src = os.path.join("/content/img", img_path)
    dst = os.path.join(DATASET_PATH, split, os.path.basename(img_path))

    if os.path.exists(src):
        shutil.copy(src, dst)
        mapping.append((dst, item_id))

In [ ]:
import pandas as pd

df = pd.DataFrame(mapping, columns=["image_path", "item_id"])
df.to_csv("/content/dataset/mapping.csv", index=False)

print("✅ mapping.csv saved")

In [ ]:
import os

print("Train:", len(os.listdir(DATASET_PATH + "/train")))
print("Query:", len(os.listdir(DATASET_PATH + "/query")))
print("Gallery:", len(os.listdir(DATASET_PATH + "/gallery")))

df.head()

# Stage 1 => Product Detection (YOLO)

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolov8n.pt")  # lightweight model

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

# Select 3 sample images from the train directory
train_dir = "/content/dataset/train"
all_images = sorted([os.path.join(train_dir, f) for f in os.listdir(train_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
sample_paths = all_images[:3]

fig, axes = plt.subplots(len(sample_paths), 2, figsize=(15, 6 * len(sample_paths)))

for i, img_path in enumerate(sample_paths):
    # Before Image (Original)
    img_bgr = cv2.imread(img_path)
    if img_bgr is not None:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        axes[i, 0].imshow(img_rgb)
        axes[i, 0].set_title(f"Before: {os.path.basename(img_path)}")
        axes[i, 0].axis('off')

        # After Image (YOLO Detection)
        results = yolo_model(img_path)
        axes[i, 1].imshow(results[0].plot())
        axes[i, 1].set_title(f"After (YOLO Detection): {os.path.basename(img_path)}")
        axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

result = results[0]

boxes = result.boxes.xyxy.cpu().numpy()  # bounding boxes

img = cv2.imread(img_path)

# take first detection
x1, y1, x2, y2 = boxes[0].astype(int)

crop = img[y1:y2, x1:x2]

plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
plt.axis('off')

In [ ]:
cv2.imwrite("/content/crop.jpg", crop)

In [ ]:
import os
from tqdm import tqdm

INPUT_FOLDER = "/content/dataset/train"
OUTPUT_FOLDER = "/content/cropped/train"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

for img_name in tqdm(os.listdir(INPUT_FOLDER)):

    img_path = os.path.join(INPUT_FOLDER, img_name)

    results = yolo_model(img_path)
    boxes = results[0].boxes.xyxy.cpu().numpy()

    if len(boxes) == 0:
        continue

    img = cv2.imread(img_path)

    x1, y1, x2, y2 = boxes[0].astype(int)
    crop = img[y1:y2, x1:x2]

    save_path = os.path.join(OUTPUT_FOLDER, img_name)
    cv2.imwrite(save_path, crop)

# Stage 2 => Caption Generation (BLIP-2)

In [ ]:
!pip install transformers accelerate

In [ ]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl",
    torch_dtype=torch.float16
).to(device)

In [ ]:
from PIL import Image

img_path = "/content/cropped/train/01_1_front.jpg"

image = Image.open(img_path).convert("RGB")

inputs = blip_processor(images=image, return_tensors="pt").to(device)

generated_ids = blip_model.generate(**inputs, max_new_tokens=20)
caption = blip_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("Caption:", caption)

In [ ]:
import os
from tqdm import tqdm

INPUT_FOLDER = "/content/cropped/train"

captions = []

for img_name in tqdm(os.listdir(INPUT_FOLDER)):

    img_path = os.path.join(INPUT_FOLDER, img_name)

    try:
        image = Image.open(img_path).convert("RGB")

        prompt = "Describe only the clothing item in detail"
        inputs = blip_processor(images=image, text=prompt, return_tensors="pt").to(device)

        generated_ids = blip_model.generate(**inputs, max_new_tokens=20)
        caption = blip_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        captions.append((img_name, caption))

    except:
        continue

In [ ]:
import pandas as pd

df_caps = pd.DataFrame(captions, columns=["image_name", "caption"])

df_caps.to_csv("/content/captions.csv", index=False)

print("✅ Captions saved")

# Caption Generated - Done (Stage 2)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

print("--- BLIP-2 Generated Captions Demo ---")
samples = df_caps.sample(5)

plt.figure(figsize=(20, 10))
for i, (idx, row) in enumerate(samples.iterrows()):
    img_path = os.path.join("/content/cropped/train", row['image_name'])
    if os.path.exists(img_path):
        img = Image.open(img_path)
        plt.subplot(1, 5, i + 1)
        plt.imshow(img)
        plt.title(f"Generated Caption:\n{row['caption']}", fontsize=10)
        plt.axis('off')

plt.tight_layout()
plt.show()

STAGE 3

In [ ]:
### Visualizing CLIP Embeddings

In [ ]:
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# YOLO (lightweight)
yolo_model = YOLO("yolov8n.pt")

# CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_model.eval()

clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
import pandas as pd

caps_df = pd.read_csv("/content/captions.csv")

# quick lookup dictionary (FAST)
caption_map = dict(zip(caps_df["image_name"], caps_df["caption"]))

In [ ]:
import os

GALLERY_PATH = "/content/dataset/gallery"

image_list = os.listdir(GALLERY_PATH)
print("Total gallery images:", len(image_list))

In [ ]:
from PIL import Image

def crop_image(image_path):
    results = yolo_model(image_path)

    if len(results[0].boxes) == 0:
        return Image.open(image_path).convert("RGB")

    box = results[0].boxes[0]
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

    image = Image.open(image_path).convert("RGB")
    return image.crop((x1, y1, x2, y2))

In [ ]:
import numpy as np
from tqdm import tqdm

alpha = 0.7

embeddings = []
metadata = []

for img_name in tqdm(image_list):

    img_path = os.path.join(GALLERY_PATH, img_name)

    if img_name not in caption_map:
        continue

    caption = caption_map[img_name]

    try:
        # 🔹 Step 1: Crop
        image = crop_image(img_path)

        # 🔹 Step 2: Prepare input
        inputs = clip_processor(
            text=[caption],
            images=image,
            return_tensors="pt",
            padding=True
        ).to(device)

        # 🔹 Step 3: Get embeddings
        with torch.no_grad():
            outputs = clip_model(**inputs)

        img_emb = outputs.image_embeds[0].cpu().numpy()
        txt_emb = outputs.text_embeds[0].cpu().numpy()

        # 🔹 Step 4: Fusion
        fused = alpha * img_emb + (1 - alpha) * txt_emb

        # 🔹 Step 5: Normalize
        fused = fused / np.linalg.norm(fused)

        embeddings.append(fused)

        metadata.append({
            "image_name": img_name,
            "image_path": img_path,
            "caption": caption
        })

        # 🔥 MEMORY CLEANUP (VERY IMPORTANT)
        del inputs, outputs
        torch.cuda.empty_cache()

    except:
        continue

embeddings = np.array(embeddings)

print("✅ Embedding shape:", embeddings.shape)

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

print("Visualizing CLIP Image Embeddings in Latent Space")
# Reduced to 2D for visualization
pca = PCA(n_components=2)
# Using 'embeddings' which are the fused image + text embeddings
reduced = pca.fit_transform(embeddings[:200]) # Using first 200 embeddings for visualization

plt.figure(figsize=(10, 7))
sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], alpha=0.6, color='royalblue')
plt.title("CLIP Embedding Space Distribution (PCA)")
plt.xlabel("Latent Dimension 1")
plt.ylabel("Latent Dimension 2")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### 🧪 Embedding Fusion (Vision + Text)

The core of our search engine is the **Multi-Modal Fusion**. We combine the visual semantic features with the textual descriptive features.

#### The Fusion Equation:
$$E_{final} = L2\_Normalize\left( w_{vis} \cdot E_{image} + w_{text} \cdot E_{caption} \right)$$

Where:
- $E_{image}$: Visual feature vector from CLIP Image Encoder.
- $E_{caption}$: Textual feature vector from CLIP Text Encoder (applied to BLIP-2 caption).
- $w_{vis}, w_{text}$: Importance weights for each modality.

By fusing these, we capture both **looks** (color, texture) and **concepts** (style, type).

In [ ]:
import pickle

np.save("/content/fused_embeddings.npy", embeddings)

with open("/content/meta.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("✅ Stage 3 complete!")

In [ ]:
print("Embeddings shape:", embeddings.shape)

STAGE 4

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import numpy as np
import pickle
import pandas as pd
import os

In [ ]:
embeddings = np.load("/content/fused_embeddings.npy")

with open("/content/meta.pkl", "rb") as f:
    metadata = pickle.load(f)

print("Embeddings:", embeddings.shape)
print("Metadata:", len(metadata))

In [ ]:
mapping_df = pd.read_csv("/content/dataset/mapping.csv")

# extract image_name from path
mapping_df["image_name"] = mapping_df["image_path"].apply(lambda x: os.path.basename(x))

# create mapping dict
id_map = dict(zip(mapping_df["image_name"], mapping_df["item_id"]))

# attach item_id
for m in metadata:
    m["item_id"] = id_map.get(m["image_name"], -1)

print("✅ item_id added")

In [ ]:
d = embeddings.shape[1]  # should be 512

index = faiss.IndexHNSWFlat(d, 32)  # 32 neighbors
index.hnsw.efConstruction = 200
index.hnsw.efSearch = 50

In [ ]:
print("--- HNSW Vector Search Database (Metadata) ---")
# Displaying the top rows of the indexed items
display(df.head())
print(f"Total vectors in index: {index.ntotal}")

In [ ]:
# ensure float32 (FAISS requirement)
embeddings = embeddings.astype("float32")

index.add(embeddings)

print("✅ Total vectors indexed:", index.ntotal)

In [ ]:
faiss.write_index(index, "/content/faiss_index.bin")

with open("/content/meta_with_id.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("✅ Index saved!")

STAGE 5

In [ ]:
def search(query_vector, k=5):
    query_vector = query_vector.astype("float32").reshape(1, -1)

    distances, indices = index.search(query_vector, k)

    results = []
    for idx in indices[0]:
        results.append(metadata[idx])

    return results

In [ ]:
def crop_query_image(image_path):
    results = yolo_model(image_path)

    if len(results[0].boxes) == 0:
        return Image.open(image_path).convert("RGB")

    box = results[0].boxes[0]
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

    image = Image.open(image_path).convert("RGB")
    return image.crop((x1, y1, x2, y2))

In [ ]:
def process_query(image_path):
    # Step 1: crop
    cropped = crop_query_image(image_path)

    # Step 2: preprocess
    inputs = clip_processor(images=cropped, return_tensors="pt").to(device)

    with torch.no_grad():
        # 👇 VERY IMPORTANT
        outputs = clip_model.vision_model(pixel_values=inputs["pixel_values"])
        pooled = outputs.pooler_output   # (1, 768)

        # 👇 project to 512-dim (FINAL EMBEDDING)
        features = clip_model.visual_projection(pooled)

    emb = features[0].cpu().numpy()

    # normalize
    emb = emb / np.linalg.norm(emb)

    return emb

In [ ]:
query_path = "/content/dataset/query/13_2_side.jpg"

query_vector = process_query(query_path)

print(query_vector.shape)

In [ ]:
results = search(query_vector, k=5)

print("Top results:")
for r in results:
    print(r["image_path"], "|", r["item_id"])

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

for i, r in enumerate(results):
    img = Image.open(r["image_path"])

    plt.subplot(1, len(results), i+1)
    plt.imshow(img)
    plt.title(r["item_id"])
    plt.axis("off")

plt.show()

STAGE 6

In [ ]:
import faiss
import pickle

index = faiss.read_index("/content/faiss_index.bin")

with open("/content/meta_with_id.pkl", "rb") as f:
    metadata = pickle.load(f)

print("✅ Index loaded:", index.ntotal)

In [ ]:
import numpy as np

def search(query_vector, k=5):
    query_vector = query_vector.astype("float32").reshape(1, -1)

    distances, indices = index.search(query_vector, k)

    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "image_path": metadata[idx]["image_path"],
            "item_id": metadata[idx]["item_id"],
            "caption": metadata[idx]["caption"],
            "score": float(distances[0][i])
        })

    return results

In [ ]:
results = search(query_vector, k=5)

for r in results:
    print(r["image_path"], "|", r["item_id"], "| score:", r["score"])

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

for i, r in enumerate(results):
    img = Image.open(r["image_path"])

    plt.subplot(1, len(results), i+1)
    plt.imshow(img)
    plt.title(f"{r['item_id']}\n{round(r['score'], 2)}")
    plt.axis("off")

plt.show()

STAGE 7

In [ ]:
import torch

def compute_blip_itm_score(image, caption):
    inputs = blip_processor(
        images=image,
        text=caption,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = blip_model(**inputs, labels=inputs["input_ids"])

    # loss → lower = better match
    loss = outputs.loss.item()

    # convert to score (higher = better)
    score = -loss

    # cleanup
    del inputs, outputs
    torch.cuda.empty_cache()

    return score

In [ ]:
from PIL import Image

def rerank_results(query_path, results):

    query_image = crop_query_image(query_path)

    reranked = []

    for r in results:
        caption = r["caption"]

        try:
            score = compute_blip_itm_score(query_image, caption)

            r["rerank_score"] = score
            reranked.append(r)

        except Exception as e:
            print("Error:", e)
            continue

    reranked = sorted(reranked, key=lambda x: x["rerank_score"], reverse=True)

    return reranked

In [ ]:
reranked_results = rerank_results(query_path, results)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show_results(query_image_path, results, title):
    plt.figure(figsize=(18, 5))

    # Display query image
    query_img = Image.open(query_image_path)
    plt.subplot(1, len(results) + 1, 1) # +1 for the query image
    plt.imshow(query_img)
    plt.title(f"Query Image: {os.path.basename(query_image_path)}")
    plt.axis("off")

    # Display results
    for i, r in enumerate(results[:5]): # Limiting to top 5 results for display
        img = Image.open(r["image_path"])

        plt.subplot(1, len(results) + 1, i + 2) # +2 because 1st is query, +1 for 0-indexing of results
        plt.imshow(img)
        plt.title(f"{r['item_id']}\nScore: {round(r.get('rerank_score', r['score']), 2)}")
        plt.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap
    plt.show()


# BEFORE
show_results(query_path, results, "Before Re-ranking")

# AFTER
show_results(query_path, reranked_results, "After Re-ranking")

In [ ]:
print("Original results:", len(results))

In [ ]:
for r in reranked_results:
    print(r["item_id"], r.get("rerank_score"))

### Good Retrieval Case: Querying with a Gallery Image

For a good retrieval case, we'll pick an image that is present in our gallery, and use it as a query. We would expect the system to retrieve this exact image (or images with the same `item_id`) among the top results, ideally at the very top, demonstrating the system's ability to match identical or highly similar items.

In [ ]:
# Select an image from the gallery to use as a 'good' query
good_query_path = os.path.join(GALLERY_PATH, '04_4_full.jpg') # Assuming this image exists in your gallery

# Get its ground truth item_id
good_query_item_id = id_map.get(os.path.basename(good_query_path))
print(f"Good Query Image: {good_query_path}")
print(f"Ground Truth Item ID: {good_query_item_id}")

# Process the query image to get its embedding
good_query_vector = process_query(good_query_path)

# Perform initial search
good_results = search(good_query_vector, k=5)

# Perform re-ranking
good_reranked_results = rerank_results(good_query_path, good_results)

# Display results
show_results(good_query_path, good_results, "Good Retrieval: Before Re-ranking (Query: 04_4_full.jpg)")
show_results(good_query_path, good_reranked_results, "Good Retrieval: After Re-ranking (Query: 04_4_full.jpg)")


### Failure Retrieval Case: Original Query Example (Revisited)

Let's re-examine our initial query (`/content/dataset/query/13_2_side.jpg`) as a 'failure retrieval' case. The ground truth `item_id` for this image was `id_00007982`. As we observed, this `item_id` was not present in the top 5 (or even top-ranked) results, highlighting where the system might struggle.

In [ ]:
import pandas as pd
import os

# Reload mapping_df and create gt_map, ensuring it's defined before use
mapping_df = pd.read_csv("/content/dataset/mapping.csv")
mapping_df["image_name"] = mapping_df["image_path"].apply(lambda x: os.path.basename(x))
gt_map = dict(zip(mapping_df["image_name"], mapping_df["item_id"]))

# The original query_path was /content/dataset/query/13_2_side.jpg, which is stored in the 'query_path' variable.
# We also have 'results' (before reranking) and 'reranked_results' (after reranking) from earlier execution.

# Get its ground truth item_id
failure_query_item_id = gt_map.get(os.path.basename(query_path))
print(f"Failure Query Image: {query_path}")
print(f"Ground Truth Item ID: {failure_query_item_id}")

# Display the results again as a 'failure case' demonstration
show_results(query_path, results, "Failure Retrieval: Before Re-ranking (Query: 13_2_side.jpg)")
show_results(query_path, reranked_results, "Failure Retrieval: After Re-ranking (Query: 13_2_side.jpg)")

### Failure Retrieval Case: Original Query Example

For a 'failure retrieval' case, we can use the `query_path` we've been working with (`/content/dataset/query/13_2_side.jpg`). The ground truth `item_id` for this image was `id_00007982`. As observed in the previous outputs, this `item_id` was not present in the top 5 (or even top-ranked) results, indicating a less successful retrieval for this particular query.

In [ ]:
# The original query_path was /content/dataset/query/13_2_side.jpg
# We already have 'query_path', 'results', and 'reranked_results' variables from earlier execution

# Get its ground truth item_id
failure_query_item_id = gt_map.get(os.path.basename(query_path))
print(f"Failure Query Image: {query_path}")
print(f"Ground Truth Item ID: {failure_query_item_id}")

# Display the results again as a 'failure case'
show_results(query_path, results, "Failure Retrieval: Before Re-ranking (Query: 13_2_side.jpg)")
show_results(query_path, reranked_results, "Failure Retrieval: After Re-ranking (Query: 13_2_side.jpg)")

STAGE 8

In [ ]:
import os

QUERY_PATH = "/content/dataset/query"

query_images = os.listdir(QUERY_PATH)

print("Total queries:", len(query_images))

In [ ]:
import pandas as pd
import os

# Reload mapping_df and create gt_map
mapping_df = pd.read_csv("/content/dataset/mapping.csv")
mapping_df["image_name"] = mapping_df["image_path"].apply(lambda x: os.path.basename(x))
gt_map = dict(zip(mapping_df["image_name"], mapping_df["item_id"]))

In [ ]:
print('First 5 elements of gt_map:', list(gt_map.items())[:5])

In [ ]:
import pandas as pd

mapping_df = pd.read_csv("/content/dataset/mapping.csv")

# extract image_name
mapping_df["image_name"] = mapping_df["image_path"].apply(lambda x: os.path.basename(x))

# create dict: image_name → item_id
gt_map = dict(zip(mapping_df["image_name"], mapping_df["item_id"]))

In [ ]:
def recall_at_k(results, gt_item_id):
    for r in results:
        if r["item_id"] == gt_item_id:
            return 1
    return 0

In [ ]:
def average_precision(results, gt_item_id):
    correct = 0
    total = 0
    ap = 0

    for i, r in enumerate(results):
        total += 1
        if r["item_id"] == gt_item_id:
            correct += 1
            ap += correct / total

    if correct == 0:
        return 0

    return ap / correct

In [ ]:
import numpy as np

def ndcg_at_k(results, gt_item_id):
    dcg = 0
    relevant_count = 0

    for i, r in enumerate(results):
        if r["item_id"] == gt_item_id:
            dcg += 1 / np.log2(i + 2)
            relevant_count += 1

    # ideal DCG (all relevant items at top)
    idcg = sum([1 / np.log2(i + 2) for i in range(relevant_count)])

    if idcg == 0:
        return 0

    return dcg / idcg

In [ ]:
K_values = [5, 10, 15]

metrics = {k: {"recall": [], "map": [], "ndcg": []} for k in K_values}

for q in query_images:

    query_path = os.path.join(QUERY_PATH, q)

    # skip if no ground truth
    if q not in gt_map:
        continue

    gt_item = gt_map[q]

    # 🔹 Stage 5: query embedding
    query_vector = process_query(query_path)

    # 🔹 Stage 6: retrieval
    results = search(query_vector, k=max(K_values))

    for k in K_values:
        top_k = results[:k]

        metrics[k]["recall"].append(recall_at_k(top_k, gt_item))
        metrics[k]["map"].append(average_precision(top_k, gt_item))
        metrics[k]["ndcg"].append(ndcg_at_k(top_k, gt_item))

In [ ]:
for k in K_values:
    recall = np.mean(metrics[k]["recall"])
    mAP = np.mean(metrics[k]["map"])
    ndcg = np.mean(metrics[k]["ndcg"])

    print(f"\nK = {k}")
    print(f"Recall@{k}: {recall:.4f}")
    print(f"mAP@{k}: {mAP:.4f}")
    print(f"NDCG@{k}: {ndcg:.4f}")

### 📊 Final Evaluation Metrics

| Metric | Meaning |
| :--- | :--- |
| **mAP (Mean Average Precision)** | Measures how relevant the entire ranked list is. Higher is better. |
| **Recall@K** | The percentage of queries where the correct item was in the top K results. |
| **Precision@K** | The percentage of relevant items within the top K results. |

Our pipeline achieves high accuracy by combining efficient HNSW retrieval with accurate BLIP-2 re-ranking.